In [178]:
import pandas as pd
import numpy as np

In [179]:
df_sq = pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\squads.csv')
df_bts=pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\batting_stats.csv')
df_bs=pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\bowling_stats.csv')
df_pt=pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\points_table.csv')
df_mt=pd.read_csv(r'C:\Users\User\Documents\IPL_predictor\data\matches.csv')

print(df_sq.isnull().sum())
print(df_bts.isnull().sum())
print(df_bs.isnull().sum())
print(df_pt.isnull().sum())
print(df_mt.isnull().sum())
#drop the row in which match was abandoned 
df_mt=df_mt[df_mt['match_result']=='completed']
print(df_mt.isnull().sum())

team_no          0
team_name        0
player           0
nationality      0
role             0
designation    247
dtype: int64
position       0
batsman        0
team           0
matches        0
innings        0
runs           0
impact         0
average        0
strike_rate    0
not_outs       0
high_score     0
balls_faced    0
hundreds       0
fifties        0
ducks          0
fours          0
sixes          0
dtype: int64
position        0
bowler          0
team            0
matches         0
innings         0
wickets         0
economy         0
impact          0
bbi             0
overs           0
balls           0
dot_balls       0
maiden_overs    0
runs            0
avg             0
4wh             0
5wh             0
dtype: int64
position     0
team         0
matches      0
wins         0
defeats      0
ties         0
abandoned    0
points       0
nrr          0
dtype: int64
match_id                0
date                    0
venue                   0
team1                   0


In [180]:
#adding a new feature to batting stats coloumn

df_bs['dotball_ratio']=df_bs['dot_balls']/df_bs['balls']




#dropping all the features not needed by us for our model
df_sq=df_sq[['team_name','player']]
df_bts=df_bts[['batsman','average','impact','strike_rate']]
df_bs=df_bs[['bowler','avg','impact','economy','dotball_ratio']]
df_mt=df_mt[['team1','team2','match_winner']]
df_pt=df_pt[['team','points','nrr']]

#features

#headtohead, teamrating,squad strength, previous match performances

df_pred=pd.DataFrame(columns=['Team Name','Team Rating','Squad Strength','Star Presence'])
#made a new df and added the 
df_pred['Team Name']=df_pt['team']
print(df_pred['Team Name'])


#headtohead of two teams dataframe 

df_headtohead=pd.DataFrame(columns=['Team1','Team2',' Winner'])


0                   Punjab Kings
1    Royal Challengers Bengaluru
2            Sunrisers Hyderabad
3               Rajasthan Royals
4                 Gujarat Titans
5            Chennai Super Kings
6                 Delhi Capitals
7          Kolkata Knight Riders
8                 Mumbai Indians
9           Lucknow Super Giants
Name: Team Name, dtype: object


In [181]:
#check if a top performer from either bowling rating or batting rating is present in the squad od one team 

#is the union of the star performers of bnatting and bowling of the entire tournament
star_players=set(df_bts['batsman']).union(set(df_bs['bowler']))


df_sq['is_star']=df_sq['player'].isin(star_players)

df_pred['Star Presence']=df_pred['Team Name'].map(df_sq.groupby('team_name')['is_star'].sum())


df_pred['Star Presence']=df_pred['Star Presence'].fillna(0)

#completed my star presence feature and now moving on to team rating and squad strength features

team_rating = (df_pt['points']*0.2 + df_pt['nrr']).set_axis(df_pt['team'])
df_pred['Team Rating']=df_pred['Team Name'].map(team_rating)
print(df_pred)

#completed my team rating feature and now moving on to squad strength feature



                     Team Name  Team Rating Squad Strength  Star Presence
0                 Punjab Kings        3.933            NaN            3.0
1  Royal Challengers Bengaluru        4.319            NaN            4.0
2          Sunrisers Hyderabad        2.815            NaN            4.0
3             Rajasthan Royals        2.602            NaN            4.0
4               Gujarat Titans        1.125            NaN            0.0
5          Chennai Super Kings        1.079            NaN            3.0
6               Delhi Capitals        0.140            NaN            0.0
7        Kolkata Knight Riders        1.751            NaN            0.0
8               Mumbai Indians        0.064            NaN            0.0
9         Lucknow Super Giants       -0.306            NaN            2.0


In [ ]:
#leaving a block here for cleaning of the data obatined after api fetches complete data of all the players
#this will be used for calculating the squad strength of each team and then adding it to the df_pred dataframe

In [ ]:
#creating a new dataset which will tell us the head to head ratings of two teams and the winner of the match between them
# Step 1: Map team abbreviations (used in df_mt) to full names (used in df_pred)
team_name_map = {
    'RCB': 'Royal Challengers Bengaluru',
    'SRH': 'Sunrisers Hyderabad',
    'MI': 'Mumbai Indians',
    'KKR': 'Kolkata Knight Riders',
    'PBKS': 'Punjab Kings',
    'RR': 'Rajasthan Royals',
    'GT': 'Gujarat Titans',
    'LSG': 'Lucknow Super Giants',
    'DC': 'Delhi Capitals',
    'CSK': 'Chennai Super Kings'
}

df_mt['team1_full'] = df_mt['team1'].map(team_name_map)
df_mt['team2_full'] = df_mt['team2'].map(team_name_map)

#make sure every abbreviation mapped successfully
print("Unmapped team1/team2 check (should all be 0):")
print(df_mt[['team1_full', 'team2_full']].isnull().sum())


# Merge match data with team1's features
df_matches_features = df_mt.merge(
    df_pred, left_on='team1_full', right_on='Team Name', how='left'
).rename(columns={
    'Team Rating': 'team1_rating',
    'Squad Strength': 'team1_squad_strength',
    'Star Presence': 'team1_star_presence'
}).drop(columns=['Team Name'])

#Merge again for team2's features
df_matches_features = df_matches_features.merge(
    df_pred, left_on='team2_full', right_on='Team Name', how='left'
).rename(columns={
    'Team Rating': 'team2_rating',
    'Squad Strength': 'team2_squad_strength',
    'Star Presence': 'team2_star_presence'
}).drop(columns=['Team Name'])


# Create the binary target variable
df_matches_features['team1_won'] = (
    df_matches_features['match_winner'] == df_matches_features['team1']
).astype(int)


#Clean up helper columns we no longer need
df_matches_features = df_matches_features.drop(columns=['team1_full', 'team2_full'])


# Final checks
print("\nShape:", df_matches_features.shape)
print("\nMissing values per column:")
print(df_matches_features.isnull().sum())
print("\nSample rows:")
print(df_matches_features.head(10))

Unmapped team1/team2 check (should all be 0):
team1_full    0
team2_full    0
dtype: int64

Shape: (38, 10)

Missing values per column:
team1                    0
team2                    0
match_winner             0
team1_rating             0
team1_squad_strength    38
team1_star_presence      0
team2_rating             0
team2_squad_strength    38
team2_star_presence      0
team1_won                0
dtype: int64

Sample rows:
  team1 team2 match_winner  team1_rating team1_squad_strength  \
0   RCB   SRH          RCB         4.319                  NaN   
1    MI   KKR           MI         0.064                  NaN   
2    RR   CSK           RR         2.602                  NaN   
3  PBKS    GT         PBKS         3.933                  NaN   
4   LSG    DC           DC        -0.306                  NaN   
5   KKR   SRH          SRH         1.751                  NaN   
6   CSK  PBKS         PBKS         1.079                  NaN   
7    DC    MI           DC         0.140       